# 🍛 Food Detection Version 3 Training (231 Categories + Multi-Dish Thali & Banana Leaf Plates)
### South Indian & Global Food Inspection AI Pipeline (V3 — Colab Auto-Save & Resume Ready)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Food_Detection/NoteBooks/Food%20Detection/Version3/food_detection_v3_training.ipynb)

---
### 🚀 What's New in Version 3:
- **Expanded 231-Class Dataset (`12,456` Real Images)**: `9,954` train / `1,247` val / `1,255` test across **231 food classes** (`0..230`, including **Poori**, **Chutney**, **Momos**, **Pani puri**, **Jalebi**, **Vada pav**, **Pav**, **Naan**, **Cutlet**, and **Poha**).
- **Real Multi-Dish Plates**: Curated Banana Leaf, Platter, and `Huggig_Real_Image_Dataset` (`v3_huggig_*`) multi-object YOLO bounding boxes.
- **Disconnect-Proof Google Drive Sync**: Trains on fast Colab local NVMe and **atomically syncs** `last.pt`, `best.pt`, `results.csv`, and `args.yaml` to Google Drive after **every single epoch** (with `last_backup.pt` fallback so mid-write disconnects never corrupt your checkpoint).
- **1-Click Auto-Resume**: If Colab disconnects during a 100-epoch run, simply re-run **Step 1 -> Step 2 -> Step 3 -> Step 5A** (or **Step 5B** directly) to continue seamlessly from the last completed epoch.

## Step 1: Install Dependencies, Verify GPU & Mount Google Drive
Mounts Google Drive upfront so every epoch's `last.pt` and `best.pt` are backed up safely.

In [ ]:
!pip install -q ultralytics torchvision opencv-python-headless pyyaml matplotlib seaborn pandas onnx

import torch, sys, os, ultralytics
print('=' * 60)
print('🖥️  SYSTEM & HARDWARE SPECIFICATIONS (V3)')
print('=' * 60)
print(f'Python Version   : {sys.version.split()[0]}')
print(f'PyTorch Version  : {torch.__version__}')
print(f'Ultralytics      : {ultralytics.__version__}')
print(f'CUDA Available   : {"✅ Yes" if torch.cuda.is_available() else "❌ No (Switch Colab runtime to GPU!)"}')
if torch.cuda.is_available():
    print(f'GPU Device Name  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print('=' * 60)
ultralytics.checks()

if 'google.colab' in str(get_ipython()):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive connected! Checkpoints will auto-sync to Drive after every epoch.')
    except Exception as e:
        print(f'Drive note: {e}')

## Step 2: Fresh GitHub Repository Clone / Pull
Clones (or updates) the repository in Colab's fast local NVMe storage (`/content/`).

In [ ]:
import os

REPO_URL = 'https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git'
REPO_NAME = 'SouthIndian_Food_Nutrition'

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(f'/content/{REPO_NAME}'):
        print(f'🚀 Cloning {REPO_NAME} from GitHub...')
        !git clone {REPO_URL} /content/{REPO_NAME}
    else:
        print(f'🔄 Updating existing {REPO_NAME}...')
        !git -C /content/{REPO_NAME} pull --ff-only
    %cd /content/{REPO_NAME}
    print(f'✅ Active Working Directory: {os.getcwd()}')
else:
    print(f'Running in local workspace: {os.getcwd()}')

## Step 3: Version 3 Dataset Configuration (231 Classes + Multi-Dish)
Configures absolute paths in `Food_Detection/DataSet/Version3 Used Data/dataset.yaml` and verifies image/label counts.

In [ ]:
import os, yaml, glob

candidate_yamls = [
    'Food_Detection/DataSet/Version3 Used Data/dataset.yaml',
    os.path.join(os.getcwd(), 'Food_Detection', 'DataSet', 'Version3 Used Data', 'dataset.yaml'),
    '/content/SouthIndian_Food_Nutrition/Food_Detection/DataSet/Version3 Used Data/dataset.yaml'
]

dataset_yaml_path = next((os.path.abspath(p) for p in candidate_yamls if os.path.exists(p)), None)
if not dataset_yaml_path:
    raise FileNotFoundError('❌ Version 3 dataset.yaml not found!')

dataset_dir = os.path.dirname(dataset_yaml_path)
with open(dataset_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = dataset_dir
data_cfg['train'] = 'images/train'
data_cfg['val'] = 'images/val'
data_cfg['test'] = 'images/test'

with open(dataset_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_cfg, f, sort_keys=False)

def count_imgs(split):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    return sum(len(glob.glob(os.path.join(dataset_dir, 'images', split, e))) for e in exts)

names = data_cfg.get('names', {})
n_tr, n_va, n_te = count_imgs('train'), count_imgs('val'), count_imgs('test')

print(f'✅ Version 3 dataset.yaml : {dataset_yaml_path}')
print(f'🔥 Total Classes (nc)    : {len(names)}')
print(f'📸 Train / Val / Test    : {n_tr} / {n_va} / {n_te} (Total: {n_tr + n_va + n_te})')

## Step 4: Multi-Dish Sample Visual Inspection
Previews multi-dish & `v3_huggig_*` plate samples with ground-truth bounding boxes.

In [ ]:
import glob, cv2, matplotlib.pyplot as plt

train_img_dir = os.path.join(dataset_dir, 'images', 'train')
train_lbl_dir = os.path.join(dataset_dir, 'labels', 'train')
multi_samples = sorted(glob.glob(os.path.join(train_img_dir, 'v3_huggig_*.jpg')))[:2] + sorted(glob.glob(os.path.join(train_img_dir, 'v3_platter_*.jpg')))[:2]
if not multi_samples:
    multi_samples = sorted(glob.glob(os.path.join(train_img_dir, '*.jpg')))[:4]

if multi_samples:
    fig, axes = plt.subplots(1, len(multi_samples), figsize=(20, 5))
    if len(multi_samples) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, multi_samples):
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(train_lbl_dir, f'{stem}.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:5])
                        x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                        x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cname = names.get(cid, str(cid))
                        cv2.putText(img, cname, (x1, max(y1 - 6, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(stem[:28], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Step 5: Disconnect-Safe Training & Auto-Resume (Version 3)
### How Drive Auto-Save & Resume Works:
1. **Fast Local Training**: Trains on Colab local NVMe (`LOCAL_RUN_DIR`) with `cache=False` and `workers=2` so 12,456 images won't hit Drive FUSE bottlenecks or RAM OOM.
2. **Atomic Per-Epoch Sync to Drive**: After **every single epoch**, a callback atomically copies `last.pt` (plus `last_backup.pt`), `best.pt`, `results.csv`, and `args.yaml` to `DRIVE_RUN_DIR` (`*.tmp` -> `os.replace`). Even if Colab disconnects mid-copy, your previous epoch's `last.pt` / `last_backup.pt` remains intact!
3. **Smart Auto-Resume**: **Step 5A** automatically detects if a valid checkpoint already exists on Google Drive and resumes from it; **Step 5B** lets you force-resume directly after a reconnect.

In [ ]:
# Shared Drive Sync & Checkpoint Utilities (run this cell before Step 5A or 5B)
import os, shutil, torch

EPOCHS = 100
BATCH_SIZE = 16
IMAGE_SIZE = 640
RUN_NAME = 'food_detection_v3_run'

HAS_DRIVE = os.path.exists('/content/drive/MyDrive')
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Food_Inspection_V3_Models/Food_Detection'
DRIVE_RUN_DIR = os.path.join(DRIVE_PROJECT_DIR, RUN_NAME)
LOCAL_PROJECT_DIR = '/content/runs/food_detection_v3' if 'google.colab' in str(get_ipython()) else 'Food_Detection/runs/train'
LOCAL_RUN_DIR = os.path.join(LOCAL_PROJECT_DIR, RUN_NAME)

os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)
if HAS_DRIVE:
    os.makedirs(os.path.join(DRIVE_RUN_DIR, 'weights'), exist_ok=True)

def atomic_copy(src, dst):
    if not os.path.exists(src):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + '.tmp'
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def sync_run_to_drive(local_dir=LOCAL_RUN_DIR, drive_dir=DRIVE_RUN_DIR):
    if not HAS_DRIVE:
        return
    w_src = os.path.join(local_dir, 'weights')
    w_dst = os.path.join(drive_dir, 'weights')
    # Keep rolling backup of previous last.pt before overwriting
    drive_last = os.path.join(w_dst, 'last.pt')
    drive_bak = os.path.join(w_dst, 'last_backup.pt')
    if os.path.exists(drive_last) and os.path.getsize(drive_last) > 1024:
        atomic_copy(drive_last, drive_bak)
    for fname in ('last.pt', 'best.pt'):
        atomic_copy(os.path.join(w_src, fname), os.path.join(w_dst, fname))
    for fname in ('results.csv', 'args.yaml', 'results.png', 'confusion_matrix.png'):
        atomic_copy(os.path.join(local_dir, fname), os.path.join(drive_dir, fname))

def register_drive_sync_callbacks(yolo_model):
    def on_epoch_end(trainer):
        try:
            sync_run_to_drive(str(trainer.save_dir), DRIVE_RUN_DIR)
            ep = getattr(trainer, 'epoch', -1) + 1
            tot = getattr(trainer, 'epochs', EPOCHS)
            if HAS_DRIVE:
                print(f'💾 [Drive Sync] Epoch {ep}/{tot} checkpoint saved safely to {DRIVE_RUN_DIR}')
        except Exception as e:
            print(f'⚠️ Drive sync warning: {e}')
    yolo_model.add_callback('on_train_epoch_end', on_epoch_end)
    yolo_model.add_callback('on_train_end', on_epoch_end)

def get_valid_drive_checkpoint():
    """Returns (ckpt_path, completed_epoch, target_epochs) or (None, 0, EPOCHS)."""
    candidates = [
        os.path.join(DRIVE_RUN_DIR, 'weights', 'last.pt'),
        os.path.join(DRIVE_RUN_DIR, 'weights', 'last_backup.pt'),
        os.path.join(LOCAL_RUN_DIR, 'weights', 'last.pt'),
    ]
    for p in candidates:
        if os.path.exists(p) and os.path.getsize(p) > 1024:
            try:
                ckpt = torch.load(p, map_location='cpu', weights_only=False)
                ep = int(ckpt.get('epoch', -1))
                tot = int(ckpt.get('train_args', {}).get('epochs', EPOCHS))
                return p, ep, tot
            except Exception as e:
                print(f'⚠️ Skipping corrupted checkpoint {p}: {e}')
    return None, 0, EPOCHS

print(f'✅ Sync helpers ready | Local: {LOCAL_RUN_DIR} | Drive: {DRIVE_RUN_DIR if HAS_DRIVE else "Not mounted"}')

### Step 5A: Smart Train / Auto-Resume (100 Epochs)
Automatically resumes from Google Drive `last.pt` if an interrupted run exists; otherwise starts fresh from `food_detection_v2_best.pt` or `yolov8s.pt`.

In [ ]:
from ultralytics import YOLO
import os, shutil, torch

ckpt_path, last_ep, tot_ep = get_valid_drive_checkpoint()

if ckpt_path is not None and last_ep >= 0 and (last_ep + 1) < tot_ep:
    print('=' * 65)
    print(f'🔄 RESUMING VERSION 3 FROM EPOCH {last_ep + 1}/{tot_ep}')
    print(f'   Source checkpoint: {ckpt_path}')
    print('=' * 65)
    os.makedirs(os.path.join(LOCAL_RUN_DIR, 'weights'), exist_ok=True)
    local_last = os.path.join(LOCAL_RUN_DIR, 'weights', 'last.pt')
    atomic_copy(ckpt_path, local_last)
    for fname in ('best.pt',):
        d_best = os.path.join(DRIVE_RUN_DIR, 'weights', fname)
        if os.path.exists(d_best):
            atomic_copy(d_best, os.path.join(LOCAL_RUN_DIR, 'weights', fname))
    for fname in ('results.csv', 'args.yaml'):
        d_f = os.path.join(DRIVE_RUN_DIR, fname)
        if os.path.exists(d_f):
            atomic_copy(d_f, os.path.join(LOCAL_RUN_DIR, fname))

    # Patch data path inside checkpoint in case Colab workspace path changed
    ckpt_data = torch.load(local_last, map_location='cpu', weights_only=False)
    if 'train_args' in ckpt_data:
        ckpt_data['train_args']['data'] = dataset_yaml_path
        ckpt_data['train_args']['project'] = LOCAL_PROJECT_DIR
        ckpt_data['train_args']['name'] = RUN_NAME
        torch.save(ckpt_data, local_last)

    model = YOLO(local_last)
    register_drive_sync_callbacks(model)
    results = model.train(resume=True)
elif ckpt_path is not None and last_ep == -1:
    print(f'✅ Training already completed ({tot_ep}/{tot_ep} epochs) at {ckpt_path}!')
    print('👉 Proceed directly to Step 6 & Step 7, or delete last.pt on Drive to start a fresh run.')
else:
    v2_candidates = [
        'Food_Detection/Models/Food Detection/Version2/food_detection_v2_best.pt',
        '/content/drive/MyDrive/Food_Inspection_V2_Models/Food_Detection/food_detection_v2_run/weights/best.pt'
    ]
    MODEL_BASE = next((p for p in v2_candidates if os.path.exists(p)), 'yolov8s.pt')
    print('=' * 65)
    print(f'🚀 LAUNCHING FRESH FOOD DETECTION VERSION 3 (Base: {MODEL_BASE})')
    print('=' * 65)

    model = YOLO(MODEL_BASE)
    register_drive_sync_callbacks(model)
    results = model.train(
        data=dataset_yaml_path,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        workers=2,
        cache=False,
        amp=True,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=LOCAL_PROJECT_DIR,
        name=RUN_NAME,
        patience=20,
        mosaic=1.0,
        mixup=0.1,
        save=True,
        save_period=5,
        exist_ok=True,
        plots=True,
        verbose=True
    )

sync_run_to_drive(LOCAL_RUN_DIR, DRIVE_RUN_DIR)
print('\n✅ Version 3 Training Finished & Synced to Google Drive!')

### Step 5B: 🔄 Quick Resume After Colab Disconnect (Standalone)
If Colab disconnected and you reconnected to a fresh runtime, run **Step 1 -> Step 2 -> Step 3 -> Sync Utilities**, then run this cell to pull `last.pt` from Google Drive and continue training.

In [ ]:
from ultralytics import YOLO
import os, torch

if 'google.colab' in str(get_ipython()) and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

ckpt_path, last_ep, tot_ep = get_valid_drive_checkpoint()
if ckpt_path is None:
    raise FileNotFoundError('❌ No valid last.pt or last_backup.pt found on Drive or local disk. Run Step 5A first.')

if last_ep == -1:
    print(f'✅ Checkpoint {ckpt_path} already finished all {tot_ep} epochs! Run Step 6 & 7 to export.')
else:
    print(f'🔄 Resuming from {ckpt_path} (completed epoch {last_ep + 1}/{tot_ep})...')
    os.makedirs(os.path.join(LOCAL_RUN_DIR, 'weights'), exist_ok=True)
    local_last = os.path.join(LOCAL_RUN_DIR, 'weights', 'last.pt')
    atomic_copy(ckpt_path, local_last)
    for fname in ('results.csv', 'args.yaml'):
        d_f = os.path.join(DRIVE_RUN_DIR, fname)
        if os.path.exists(d_f):
            atomic_copy(d_f, os.path.join(LOCAL_RUN_DIR, fname))

    ckpt_data = torch.load(local_last, map_location='cpu', weights_only=False)
    if 'train_args' in ckpt_data:
        ckpt_data['train_args']['data'] = dataset_yaml_path
        ckpt_data['train_args']['project'] = LOCAL_PROJECT_DIR
        ckpt_data['train_args']['name'] = RUN_NAME
        torch.save(ckpt_data, local_last)

    resume_model = YOLO(local_last)
    register_drive_sync_callbacks(resume_model)
    results = resume_model.train(resume=True)
    sync_run_to_drive(LOCAL_RUN_DIR, DRIVE_RUN_DIR)
    print('✅ Resumed training completed & synced to Drive!')

## Step 6: Analytical Plots & Metric Visualization

In [ ]:
import os, matplotlib.pyplot as plt, matplotlib.image as mpimg

run_dir = LOCAL_RUN_DIR if os.path.exists(os.path.join(LOCAL_RUN_DIR, 'results.png')) else DRIVE_RUN_DIR

for title, fname, sz in [
    ('YOLOv8 V3 Training Curves', 'results.png', (18, 9)),
    ('YOLOv8 V3 Confusion Matrix', 'confusion_matrix.png', (16, 16)),
]:
    p = os.path.join(run_dir, fname)
    if os.path.exists(p):
        plt.figure(figsize=sz)
        plt.imshow(mpimg.imread(p))
        plt.title(title, fontsize=15)
        plt.axis('off')
        plt.show()

## Step 7 & 8: Export `food_detection_v3_best.pt` / `.onnx` & 1-Click Download

In [ ]:
import os, shutil
from ultralytics import YOLO

best_candidates = [
    os.path.join(LOCAL_RUN_DIR, 'weights', 'best.pt'),
    os.path.join(DRIVE_RUN_DIR, 'weights', 'best.pt'),
    os.path.join(DRIVE_RUN_DIR, 'weights', 'last.pt'),
]
best_src = next((p for p in best_candidates if os.path.exists(p)), None)
out_dir = 'Food_Detection/Models/Food Detection/Version3'
os.makedirs(out_dir, exist_ok=True)

if best_src:
    v3_pt = os.path.join(out_dir, 'food_detection_v3_best.pt')
    shutil.copy2(best_src, v3_pt)
    shutil.copy2(best_src, 'food_detection_v3_best.pt')
    if HAS_DRIVE:
        atomic_copy(best_src, os.path.join(DRIVE_PROJECT_DIR, 'food_detection_v3_best.pt'))
    print(f'🏆 Saved: {v3_pt} (from {best_src})')
    try:
        onnx_f = YOLO(v3_pt).export(format='onnx', imgsz=640, simplify=True)
        if os.path.exists(onnx_f):
            shutil.copy2(onnx_f, os.path.join(out_dir, 'food_detection_v3_best.onnx'))
            shutil.copy2(onnx_f, 'food_detection_v3_best.onnx')
    except Exception as e:
        print(f'ONNX note: {e}')

    try:
        from google.colab import files
        files.download('food_detection_v3_best.pt')
        if os.path.exists('food_detection_v3_best.onnx'):
            files.download('food_detection_v3_best.onnx')
    except ImportError:
        pass
else:
    print('❌ best.pt not found yet.')